In [26]:
import numpy as np
from scipy.sparse import coo_array, permute_dims
from peoplegator_namedfaces.clustering.evaluation.src.utils import (
    load_clustering_files,
    load_ground_truth_indexers,
)
import pathlib

In [42]:
dataset_root = pathlib.Path("/mnt/backup/HistoricalNERProcessing/notebooks")
predictions_root = pathlib.Path(
    "/mnt/backup/PeopleGatorNamedFaces/people-gator-named-faces-baselines/.embeddings"
)
date="2026-02-11"
(
    ground_truth,
    predicted_clusters,
    _unique_faces,
) = load_clustering_files(
    ground_truths_path=dataset_root / f"people_gator__corresponding_faces__{date}.jsonl",
    predictions_path=predictions_root / "vit_small_patch8_gap_112.cosface_ms1mv3" / "clusters.agglomerative.jsonl",
    unique_faces_path=dataset_root / f"people_gator__unique_faces__{date}.jsonl",
)
(
    unique_faces, unique_person_names, unique_annotators, 
    _, _, _, 
    face_indices, person_name_indices, annotator_indices
) = load_ground_truth_indexers(ground_truth, _unique_faces)
# create a sparse matrix of shape [Fa, Na, An] where values represent assignment of face to name by annotator
face_name_annotator_matrix = coo_array(
    (np.ones_like(face_indices, dtype=int), (
        face_indices, person_name_indices, annotator_indices
    )), 
    shape=(
        len(unique_faces), 
        len(unique_person_names), 
        len(unique_annotators)
    ), 
    dtype=int
)
face_name_annotator_matrix.sum_duplicates()
face_name_annotator_matrix = face_name_annotator_matrix.minimum(1)
face_name_annotator_matrix

<COOrdinate sparse array of dtype 'int64'
	with 6752 stored elements and shape (27704, 647, 4)>

In [43]:
face_name_annotator_matrix.shape

(27704, 647, 4)

In [45]:
name_correspondence_matrix = coo_array((face_name_annotator_matrix.data, (face_name_annotator_matrix.coords[0], face_name_annotator_matrix.coords[1])), shape=(
    len(unique_faces), len(unique_person_names)), dtype=int)
name_correspondence_matrix.sum_duplicates()
# [Fa, Na, An]
# [Fa, Na, An].permute(2, 0, 1) -> [An, Fa, Na]
# [Fa, Na, An].permute(2, 1, 0) -> [An, Na, Fa]
# [An, Fa, Na] x [An, Na, Fa] -> [An, Fa, Fa] values in <0, Na> if errors are present
# [An, Fa, Fa].sum(axis=0) -> [Fa, Fa]
# [Fa, Fa] -> how many times has each pair of faces been assigned the same name by different annotators?
adjencency_matrices: coo_array = permute_dims(face_name_annotator_matrix, (2, 0, 1), copy=True) @ permute_dims(face_name_annotator_matrix, (2, 1, 0), copy=True)
adjencency_matrices = adjencency_matrices.minimum(1)
adjencency_agreement = coo_array((adjencency_matrices.data, (adjencency_matrices.row, adjencency_matrices.col)), shape=(len(unique_faces), len(unique_faces)), dtype=int)
adjencency_agreement.sum_duplicates()
adjencency_agreement

<COOrdinate sparse array of dtype 'int64'
	with 102865 stored elements and shape (27704, 27704)>

In [ ]:
i = name_correspondence_matrix.min()

np.int64(0)